# ⚡ Wickless Candle Scalping Strategy — BTCUSDT

**Strategy Logic:**
- Detect *wickless* candles (no top wick = bearish signal, no bottom wick = bullish signal)
- Filter by displacement (body-to-range ratio)
- Determine trend on higher timeframe using Supertrend + EMA slope
- Enter on lower timeframe wickless candles **aligned with higher-TF trend**
- Stop: min(nearest swing level, ATR-based stop)
- Target: fixed 2:1 RR (default) or trailing stop

---

### Mathematical Framework

**Wickless Detection:**
```
For BULLISH wickless (no bottom wick):
  bottom_wick = open - low           (for bullish candle: open < close)
  bottom_wick = close - low          (for bearish candle: close < open)
  → generic: bottom_wick = min(open, close) - low
  Condition: bottom_wick <= wick_tolerance

For BEARISH wickless (no top wick):
  top_wick = high - max(open, close)
  Condition: top_wick <= wick_tolerance
```

**Displacement (Body-to-Range Ratio):**
```
B2R = |close - open| / (high - low)
Condition: B2R >= displacement_threshold
```

**Supertrend (Trend Detection):**
```
TR = max(high - low, |high - prev_close|, |low - prev_close|)
ATR_n = EMA(TR, n)
Basic Upper Band = (high + low)/2 + multiplier * ATR_n
Basic Lower Band = (high + low)/2 - multiplier * ATR_n
Supertrend flips to bullish when close > Upper Band, bearish when close < Lower Band
```

**Stop Loss:**
```
SL_ATR   = entry ± (atr_sl_multiplier × ATR)
SL_Swing = nearest swing low (bullish) or swing high (bearish) within lookback window
SL       = min(|entry - SL_ATR|, |entry - SL_Swing|)  → tightest stop
```

**Target (2:1 RR default):**
```
TP = entry + rr_ratio * |entry - SL|   (bullish)
TP = entry - rr_ratio * |entry - SL|   (bearish)
```

**Trailing Stop (optional):**
```
Trail distance = trailing_atr_mult × ATR
For long:  trail_stop = max(trail_stop, close - trail_dist)  each bar
For short: trail_stop = min(trail_stop, close + trail_dist)  each bar
```

---
## Cell 1 — Install & Imports

In [ ]:
# ── Install dependencies (run once on Colab/Kaggle) ──────────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# Pin compatible numpy + scipy to avoid numpy._core.umath conflict on Kaggle/Colab
install("numpy==1.26.4")
install("scipy==1.13.1")
install("yfinance")
install("pandas-ta")
install("matplotlib")
install("seaborn")

# Force reload numpy with the pinned version (needed in Colab/Kaggle)
import importlib, numpy
importlib.reload(numpy)

# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_ta as ta
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── scipy stats — graceful fallback if version conflict persists ──────────────
try:
    from scipy import stats as _stats
    spearmanr   = _spearmanr
    ttest_1samp = _ttest_1samp
    _scipy_ok   = True
    print("\u2705 scipy loaded successfully")
except ImportError:
    _scipy_ok = False
    print("\u26a0\ufe0f  scipy unavailable \u2014 using numpy fallback for stats")

    def spearmanr(x, y):
        """Manual Spearman correlation via rank."""
        x, y = np.array(x), np.array(y)
        n = len(x)
        rx = np.argsort(np.argsort(x)).astype(float)
        ry = np.argsort(np.argsort(y)).astype(float)
        d2 = np.sum((rx - ry) ** 2)
        rho = 1 - (6 * d2) / (n * (n**2 - 1))
        t_stat = rho * np.sqrt((n - 2) / (1 - rho**2 + 1e-12))
        from math import erfc, sqrt
        p = 2 * (1 - 0.5 * erfc(-abs(t_stat) / sqrt(2)))
        return rho, p

    def ttest_1samp(a, popmean):
        """Manual one-sample t-test."""
        a = np.array(a)
        n = len(a)
        mean = np.mean(a)
        se   = np.std(a, ddof=1) / np.sqrt(n)
        t    = (mean - popmean) / (se + 1e-12)
        from math import erfc, sqrt
        p = 2 * (1 - 0.5 * erfc(-abs(t) / sqrt(2)))
        return t, p

print("\u2705 All imports successful")


---
## Cell 2 — ⚙️ STRATEGY PARAMETERS (Edit Here)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#                        ⚙️  STRATEGY PARAMETERS
#         Edit all values here. Everything else runs automatically.
# ══════════════════════════════════════════════════════════════════════════════

# ── DATA SOURCE ───────────────────────────────────────────────────────────────
# To use yfinance (default): set USE_LOCAL_DATA = False
# To use your own CSV file: set USE_LOCAL_DATA = True and provide path below
USE_LOCAL_DATA       = False
LOCAL_DATA_PATH      = "/path/to/your/BTCUSDT_1m_data.csv"  # ← Paste your file path here
LOCAL_DATA_DATETIME_COL = "open_time"   # column name for datetime index in your CSV

# ── SYMBOL & DATE RANGE (used only if USE_LOCAL_DATA = False) ─────────────────
SYMBOL               = "BTC-USD"        # yfinance symbol
START_DATE           = "2024-01-01"
END_DATE             = "2024-12-31"

# ── TIMEFRAMES ────────────────────────────────────────────────────────────────
LOWER_TF             = "1m"             # Execution / entry timeframe
HIGHER_TF            = "15m"            # Trend / regime timeframe
# Valid values: "1m", "5m", "15m", "30m", "1h", "4h", "1d"

# ── WICKLESS CANDLE DEFINITION ────────────────────────────────────────────────
# 0 = strict zero wick (price must equal open or close exactly)
# Set to small $ value to allow a few ticks, e.g. 0.5 for 50 cents tolerance
WICK_TOLERANCE       = 0.0              # in price units (USD)

# ── DISPLACEMENT FILTER ───────────────────────────────────────────────────────
USE_DISPLACEMENT     = True             # True = require displacement; False = wickless candle alone
DISPLACEMENT_THRESHOLD = 0.75          # Body-to-Range ratio: |close-open|/(high-low)
                                        # 0.75 = body must be 75%+ of full candle range

# ── TREND DETECTION (Higher TF) ───────────────────────────────────────────────
# Supertrend parameters
SUPERTREND_PERIOD    = 10               # ATR period for Supertrend
SUPERTREND_MULT      = 3.0              # Multiplier for Supertrend bands

# EMA slope confirmation (secondary filter alongside Supertrend)
EMA_PERIOD           = 50              # EMA period for slope confirmation
EMA_SLOPE_LOOKBACK   = 3               # Number of bars to measure EMA slope over
REQUIRE_EMA_CONFIRM  = True            # True = both Supertrend AND EMA slope must agree

# ── ENTRY MODE ────────────────────────────────────────────────────────────────
# 'close'  = enter at close of the wickless candle (default, aggressive)
# 'retest' = wait for price to pull back into the candle body before entering
ENTRY_MODE           = 'close'         # Options: 'close' | 'retest'

# If ENTRY_MODE = 'retest': how many bars to wait for the retest before cancelling
RETEST_TIMEOUT_BARS  = 5               # Cancel trade if no retest within N bars

# ── STOP LOSS ─────────────────────────────────────────────────────────────────
ATR_PERIOD           = 14              # ATR period for stop calculation
ATR_SL_MULTIPLIER    = 1.5             # SL = entry ± (ATR_SL_MULTIPLIER × ATR)
SWING_LOOKBACK       = 20              # Bars to look back for nearest swing high/low
# Final SL = min(ATR-based distance, swing-based distance) — tightest stop wins

# ── TAKE PROFIT / TRAILING ────────────────────────────────────────────────────
USE_TRAILING_STOP    = False           # True = trailing stop; False = fixed RR
RR_RATIO             = 2.0             # Take profit = entry ± RR_RATIO × SL_distance (if not trailing)
TRAILING_ATR_MULT    = 2.0             # Trail distance = TRAILING_ATR_MULT × ATR (if trailing)

# ── POSITION SIZING & RISK ────────────────────────────────────────────────────
INITIAL_CAPITAL      = 10_000          # USD
RISK_PER_TRADE_PCT   = 1.0             # % of capital to risk per trade
COMMISSION_PCT       = 0.05            # Round-trip commission % (e.g. 0.05 = 5bps each way)

# ── DISPLAY ───────────────────────────────────────────────────────────────────
SHOW_TRADE_LOG       = True            # Print trade-by-trade log
MAX_TRADES_IN_LOG    = 50              # Cap on rows displayed in trade log table

print("⚙️  Parameters loaded.")
print(f"   Timeframes : {LOWER_TF} (entry) | {HIGHER_TF} (trend)")
print(f"   Entry mode : {ENTRY_MODE}")
print(f"   Wick tol   : {WICK_TOLERANCE} USD")
print(f"   Displacement: {'ON (threshold=' + str(DISPLACEMENT_THRESHOLD) + ')' if USE_DISPLACEMENT else 'OFF'}")
print(f"   Trailing   : {'ON' if USE_TRAILING_STOP else 'OFF (RR=' + str(RR_RATIO) + ':1)'}")

---
## Cell 3 — Data Loading

In [ ]:
# ── Timeframe string → pandas resample rule ───────────────────────────────────
TF_MAP = {
    "1m": "1min", "5m": "5min", "15m": "15min", "30m": "30min",
    "1h": "1h",   "4h": "4h",   "1d": "1D"
}

def resample_ohlcv(df: pd.DataFrame, rule: str) -> pd.DataFrame:
    """Resample a 1-min OHLCV dataframe to any higher timeframe."""
    agg = {
        'Open':   'first',
        'High':   'max',
        'Low':    'min',
        'Close':  'last',
        'Volume': 'sum'
    }
    return df.resample(rule).agg(agg).dropna()


# ── Load base 1-min data ──────────────────────────────────────────────────────
if USE_LOCAL_DATA:
    print(f"📂 Loading local data from: {LOCAL_DATA_PATH}")
    df_raw = pd.read_csv(LOCAL_DATA_PATH, parse_dates=[LOCAL_DATA_DATETIME_COL])
    df_raw = df_raw.set_index(LOCAL_DATA_DATETIME_COL)
    df_raw.index = pd.to_datetime(df_raw.index, utc=True)
    # Standardise column names to Title Case
    df_raw.columns = [c.strip().title() for c in df_raw.columns]
    required_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    df_raw = df_raw[required_cols]
    print(f"   Loaded {len(df_raw):,} rows from {df_raw.index[0]} to {df_raw.index[-1]}")

else:
    print(f"🌐 Fetching {SYMBOL} 1m data from yfinance [{START_DATE} → {END_DATE}]")
    print("   Note: yfinance limits 1m data to last 7 days; for longer range use '5m' or provide local file.")
    
    # Try 1m first; fall back to 5m for longer date ranges
    try:
        df_raw = yf.download(SYMBOL, start=START_DATE, end=END_DATE,
                             interval=LOWER_TF, auto_adjust=True, progress=False)
        if len(df_raw) < 100:
            raise ValueError("Too few rows — yfinance 1m limited to 7 days")
    except Exception as e:
        print(f"   ⚠️  {e}")
        print("   → Falling back to 5m data (set USE_LOCAL_DATA=True for full 1m history)")
        LOWER_TF = "5m"
        df_raw = yf.download(SYMBOL, start=START_DATE, end=END_DATE,
                             interval="5m", auto_adjust=True, progress=False)
    
    df_raw.columns = df_raw.columns.droplevel(1) if isinstance(df_raw.columns, pd.MultiIndex) else df_raw.columns
    df_raw.index = pd.to_datetime(df_raw.index, utc=True)

print(f"\n✅ Base data: {len(df_raw):,} bars | {df_raw.index[0]} → {df_raw.index[-1]}")
print(f"   Columns: {list(df_raw.columns)}")


# ── Build lower TF and higher TF dataframes ───────────────────────────────────
# Lower TF = the raw data (already at LOWER_TF resolution)
df_ltf = df_raw.copy()

# Higher TF = resample
htf_rule = TF_MAP.get(HIGHER_TF, HIGHER_TF)
df_htf   = resample_ohlcv(df_ltf, htf_rule)

print(f"   LTF ({LOWER_TF}): {len(df_ltf):,} bars")
print(f"   HTF ({HIGHER_TF}): {len(df_htf):,} bars")
df_ltf.tail(3)

---
## Cell 4 — Indicator Engine

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  INDICATOR ENGINE
# ══════════════════════════════════════════════════════════════════════════════

# ── 1. ATR (Wilder's smoothing) ───────────────────────────────────────────────
def compute_atr(df: pd.DataFrame, period: int) -> pd.Series:
    """True Range → ATR using Wilder's EMA (same as TradingView)."""
    hi, lo, cl = df['High'], df['Low'], df['Close']
    prev_cl = cl.shift(1)
    tr = pd.concat([
        hi - lo,
        (hi - prev_cl).abs(),
        (lo - prev_cl).abs()
    ], axis=1).max(axis=1)
    atr = tr.ewm(alpha=1/period, adjust=False).mean()
    return atr


# ── 2. Supertrend ─────────────────────────────────────────────────────────────
def compute_supertrend(df: pd.DataFrame, period: int, multiplier: float):
    """
    Returns:
        supertrend (Series): actual supertrend line value
        direction  (Series): +1 = bullish (price above ST), -1 = bearish
    """
    hi, lo, cl = df['High'], df['Low'], df['Close']
    atr = compute_atr(df, period)
    hl2 = (hi + lo) / 2

    basic_upper = hl2 + multiplier * atr
    basic_lower = hl2 - multiplier * atr

    upper = basic_upper.copy()
    lower = basic_lower.copy()
    supertrend = pd.Series(np.nan, index=df.index)
    direction  = pd.Series(0, index=df.index)

    for i in range(1, len(df)):
        # Final upper band
        if basic_upper.iloc[i] < upper.iloc[i-1] or cl.iloc[i-1] > upper.iloc[i-1]:
            upper.iloc[i] = basic_upper.iloc[i]
        else:
            upper.iloc[i] = upper.iloc[i-1]

        # Final lower band
        if basic_lower.iloc[i] > lower.iloc[i-1] or cl.iloc[i-1] < lower.iloc[i-1]:
            lower.iloc[i] = basic_lower.iloc[i]
        else:
            lower.iloc[i] = lower.iloc[i-1]

        # Direction
        if supertrend.iloc[i-1] == upper.iloc[i-1]:
            direction.iloc[i] = -1 if cl.iloc[i] <= upper.iloc[i] else 1
        else:
            direction.iloc[i] = 1 if cl.iloc[i] >= lower.iloc[i] else -1

        supertrend.iloc[i] = lower.iloc[i] if direction.iloc[i] == 1 else upper.iloc[i]

    return supertrend, direction


# ── 3. EMA Slope ──────────────────────────────────────────────────────────────
def compute_ema_slope(df: pd.DataFrame, period: int, lookback: int) -> pd.Series:
    """
    Returns slope direction: +1 (rising), -1 (falling)
    Based on EMA[0] vs EMA[lookback] bars ago.
    """
    ema = df['Close'].ewm(span=period, adjust=False).mean()
    slope = np.sign(ema - ema.shift(lookback))
    return slope.astype(int), ema


# ── 4. Wickless Candle Detection ──────────────────────────────────────────────
def detect_wickless(df: pd.DataFrame, tolerance: float):
    """
    bottom_wick = min(open, close) - low   → bullish wickless if <= tolerance
    top_wick    = high - max(open, close)  → bearish wickless if <= tolerance

    A candle can simultaneously be bullish-wickless (no bottom wick)
    AND bearish-wickless (no top wick) — e.g. a full-body candle (marubozu).
    We tag both; direction filter is applied during signal generation.
    """
    bottom_wick = df[['Open','Close']].min(axis=1) - df['Low']
    top_wick    = df['High'] - df[['Open','Close']].max(axis=1)

    bullish_wickless = (bottom_wick <= tolerance) & (df['Close'] > df['Open'])
    bearish_wickless = (top_wick    <= tolerance) & (df['Close'] < df['Open'])

    return bullish_wickless, bearish_wickless, bottom_wick, top_wick


# ── 5. Body-to-Range Ratio (Displacement) ─────────────────────────────────────
def compute_b2r(df: pd.DataFrame) -> pd.Series:
    """
    B2R = |close - open| / (high - low)
    Handles division by zero (doji) by returning 0.
    """
    body  = (df['Close'] - df['Open']).abs()
    range_ = df['High'] - df['Low']
    b2r = body / range_.replace(0, np.nan)
    return b2r.fillna(0)


# ── 6. Swing High / Low Detection ─────────────────────────────────────────────
def compute_swings(df: pd.DataFrame, lookback: int):
    """
    For each bar: find nearest swing low (support) and swing high (resistance)
    within the last `lookback` bars.
    Uses a local minima/maxima approach (non-overlapping window).
    """
    n = len(df)
    swing_low  = pd.Series(np.nan, index=df.index)
    swing_high = pd.Series(np.nan, index=df.index)

    lows  = df['Low'].values
    highs = df['High'].values

    for i in range(lookback, n):
        window_low  = lows[i-lookback:i]
        window_high = highs[i-lookback:i]
        swing_low.iloc[i]  = np.min(window_low)
        swing_high.iloc[i] = np.max(window_high)

    return swing_low, swing_high


print("✅ Indicator functions defined")

---
## Cell 5 — Apply Indicators to HTF & LTF

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  APPLY INDICATORS
# ══════════════════════════════════════════════════════════════════════════════

# ── Higher TF: Supertrend + EMA slope ─────────────────────────────────────────
print(f"Computing HTF ({HIGHER_TF}) indicators...")
st_line, st_dir = compute_supertrend(df_htf, SUPERTREND_PERIOD, SUPERTREND_MULT)
ema_slope_htf, ema_line_htf = compute_ema_slope(df_htf, EMA_PERIOD, EMA_SLOPE_LOOKBACK)

df_htf['supertrend']     = st_line
df_htf['st_direction']   = st_dir          # +1 bull, -1 bear
df_htf['ema']            = ema_line_htf
df_htf['ema_slope']      = ema_slope_htf   # +1 rising, -1 falling

# Combined HTF trend: Supertrend + EMA slope must both agree (if REQUIRE_EMA_CONFIRM)
if REQUIRE_EMA_CONFIRM:
    df_htf['htf_trend'] = np.where(
        (df_htf['st_direction'] == 1)  & (df_htf['ema_slope'] == 1),  1,
        np.where(
        (df_htf['st_direction'] == -1) & (df_htf['ema_slope'] == -1), -1, 0)
    )
else:
    df_htf['htf_trend'] = df_htf['st_direction']

print(f"   HTF trend distribution: {df_htf['htf_trend'].value_counts().to_dict()}")


# ── Lower TF: All signals ──────────────────────────────────────────────────────
print(f"\nComputing LTF ({LOWER_TF}) indicators...")

# ATR for SL
df_ltf['atr'] = compute_atr(df_ltf, ATR_PERIOD)

# Wickless detection
bull_wl, bear_wl, bot_wick, top_wick = detect_wickless(df_ltf, WICK_TOLERANCE)
df_ltf['bull_wickless'] = bull_wl
df_ltf['bear_wickless'] = bear_wl
df_ltf['bottom_wick']   = bot_wick
df_ltf['top_wick']      = top_wick

# Displacement
df_ltf['b2r'] = compute_b2r(df_ltf)
df_ltf['displaced'] = df_ltf['b2r'] >= DISPLACEMENT_THRESHOLD if USE_DISPLACEMENT else True

# Swing levels for SL
df_ltf['swing_low'],  df_ltf['swing_high'] = compute_swings(df_ltf, SWING_LOOKBACK)

print(f"   Bullish wickless candles : {bull_wl.sum():,}")
print(f"   Bearish wickless candles : {bear_wl.sum():,}")
print(f"   Displaced (B2R≥{DISPLACEMENT_THRESHOLD}): {df_ltf['displaced'].sum():,}")
print(f"   Bull wickless + displaced: {(bull_wl & df_ltf['displaced']).sum():,}")
print(f"   Bear wickless + displaced: {(bear_wl & df_ltf['displaced']).sum():,}")


# ── Merge HTF trend into LTF (merge_asof — forward fill HTF into LTF bars) ───
print("\nAligning HTF trend onto LTF bars (merge_asof)...")
htf_trend_df = df_htf[['htf_trend', 'st_direction', 'ema_slope']].reset_index()
htf_trend_df.columns = ['timestamp', 'htf_trend', 'st_direction', 'ema_slope']

ltf_reset = df_ltf.reset_index()
ltf_reset.columns.values[0] = 'timestamp'

merged = pd.merge_asof(
    ltf_reset.sort_values('timestamp'),
    htf_trend_df.sort_values('timestamp'),
    on='timestamp',
    direction='backward'       # Use the LAST COMPLETED HTF bar's trend
)
merged = merged.set_index('timestamp')
df_ltf = merged

print(f"✅ LTF df shape after merge: {df_ltf.shape}")
df_ltf[['Close','atr','b2r','bull_wickless','bear_wickless','htf_trend']].tail(5)

---
## Cell 6 — Signal Generation

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  SIGNAL GENERATION
#  Long  signal: bullish wickless + displaced + HTF trend = +1
#  Short signal: bearish wickless + displaced + HTF trend = -1
# ══════════════════════════════════════════════════════════════════════════════

def generate_signals(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    displacement_filter = df['displaced'] if USE_DISPLACEMENT else pd.Series(True, index=df.index)
    
    # Raw signal condition
    long_cond  = df['bull_wickless'] & displacement_filter & (df['htf_trend'] == 1)
    short_cond = df['bear_wickless'] & displacement_filter & (df['htf_trend'] == -1)

    df['raw_signal'] = 0
    df.loc[long_cond,  'raw_signal'] =  1
    df.loc[short_cond, 'raw_signal'] = -1

    print(f"   Raw long  signals: {long_cond.sum():,}")
    print(f"   Raw short signals: {short_cond.sum():,}")
    print(f"   Total raw signals: {(long_cond | short_cond).sum():,}")
    return df

df_ltf = generate_signals(df_ltf)
print("\n✅ Signals generated")

---
## Cell 7 — Backtest Engine

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  BACKTEST ENGINE
#  Simulates bar-by-bar execution with:
#  - Entry: 'close' or 'retest' mode
#  - SL: min(ATR-based, nearest swing) — tightest wins
#  - TP: fixed RR or trailing ATR stop
#  - Position sizing: risk-based (risk_pct of capital / SL distance)
#  - Commission: applied on entry and exit
# ══════════════════════════════════════════════════════════════════════════════

def run_backtest(df: pd.DataFrame) -> tuple:
    """
    Returns:
        trades (list of dicts): trade log
        equity_curve (pd.Series): equity at each bar
    """
    capital    = INITIAL_CAPITAL
    risk_amt   = capital * (RISK_PER_TRADE_PCT / 100)
    commission = COMMISSION_PCT / 100

    trades = []
    equity = []

    # State
    in_trade        = False
    direction       = 0
    entry_price     = 0.0
    sl_price        = 0.0
    tp_price        = 0.0
    trail_stop      = 0.0
    position_size   = 0.0     # In BTC units
    entry_time      = None
    entry_idx       = 0
    sl_distance     = 0.0

    # Retest mode state
    pending_signal  = 0
    pending_body_hi = 0.0
    pending_body_lo = 0.0
    pending_sl      = 0.0
    pending_atr     = 0.0
    pending_bar     = 0

    close_arr  = df['Close'].values
    high_arr   = df['High'].values
    low_arr    = df['Low'].values
    open_arr   = df['Open'].values
    signal_arr = df['raw_signal'].values
    atr_arr    = df['atr'].values
    swl_arr    = df['swing_low'].values
    swh_arr    = df['swing_high'].values
    idx_arr    = df.index

    def compute_sl_distance(entry, sig, atr, swing_l, swing_h):
        atr_dist   = ATR_SL_MULTIPLIER * atr
        if sig == 1:   # long
            swing_dist = entry - swing_l if not np.isnan(swing_l) else atr_dist
        else:          # short
            swing_dist = swing_h - entry if not np.isnan(swing_h) else atr_dist
        # Take the TIGHTER of the two
        dist = min(atr_dist, max(swing_dist, 1e-6))  # floor at epsilon
        return max(dist, 1e-6)  # always positive

    for i in range(1, len(df)):
        c = close_arr[i]
        h = high_arr[i]
        l = low_arr[i]
        o = open_arr[i]
        atr_val  = atr_arr[i]
        sig      = signal_arr[i]

        equity.append(capital)

        # ── Manage open trade ────────────────────────────────────────────────
        if in_trade:
            hit_sl = False
            hit_tp = False
            exit_p = np.nan

            if USE_TRAILING_STOP:
                trail_dist = TRAILING_ATR_MULT * atr_val
                if direction == 1:
                    trail_stop = max(trail_stop, c - trail_dist)
                    if l <= trail_stop:
                        hit_sl = True; exit_p = trail_stop
                else:
                    trail_stop = min(trail_stop, c + trail_dist)
                    if h >= trail_stop:
                        hit_sl = True; exit_p = trail_stop
            else:
                if direction == 1:
                    if l <= sl_price:   hit_sl = True; exit_p = sl_price
                    elif h >= tp_price: hit_tp = True; exit_p = tp_price
                else:
                    if h >= sl_price:   hit_sl = True; exit_p = sl_price
                    elif l <= tp_price: hit_tp = True; exit_p = tp_price

            if hit_sl or hit_tp:
                pnl_pts  = (exit_p - entry_price) * direction
                pnl_usd  = pnl_pts * position_size
                comm_usd = commission * (entry_price + exit_p) * position_size
                net_pnl  = pnl_usd - comm_usd
                capital += net_pnl

                trades.append({
                    'entry_time'    : entry_time,
                    'exit_time'     : idx_arr[i],
                    'direction'     : 'LONG' if direction == 1 else 'SHORT',
                    'entry_price'   : round(entry_price, 2),
                    'exit_price'    : round(exit_p, 2),
                    'sl_price'      : round(sl_price if not USE_TRAILING_STOP else trail_stop, 2),
                    'tp_price'      : round(tp_price, 2) if not USE_TRAILING_STOP else np.nan,
                    'sl_distance'   : round(sl_distance, 2),
                    'position_size' : round(position_size, 6),
                    'pnl_usd'       : round(net_pnl, 2),
                    'pnl_pct'       : round(net_pnl / (capital - net_pnl) * 100, 3),
                    'result'        : 'WIN' if net_pnl > 0 else 'LOSS',
                    'bars_held'     : i - entry_idx,
                    'exit_reason'   : 'TP' if hit_tp else ('TRAIL_SL' if USE_TRAILING_STOP else 'SL'),
                    'capital_after' : round(capital, 2)
                })
                in_trade = False
            continue

        # ── Handle pending retest entry ───────────────────────────────────────
        if ENTRY_MODE == 'retest' and pending_signal != 0:
            bars_since = i - pending_bar
            entered = False

            if pending_signal == 1:
                # Price retraces INTO the bullish candle body (between open and close)
                if l <= pending_body_hi and h >= pending_body_lo:
                    entry_price = pending_body_hi  # enter at top of body on retest
                    entered = True
            else:
                if h >= pending_body_lo and l <= pending_body_hi:
                    entry_price = pending_body_lo
                    entered = True

            if entered:
                direction = pending_signal
                sl_dist   = compute_sl_distance(entry_price, direction,
                                                pending_atr, swl_arr[i], swh_arr[i])
                sl_distance   = sl_dist
                sl_price      = entry_price - direction * sl_dist
                tp_price      = entry_price + direction * RR_RATIO * sl_dist
                risk_amt      = capital * (RISK_PER_TRADE_PCT / 100)
                position_size = risk_amt / sl_dist
                trail_stop    = entry_price - direction * TRAILING_ATR_MULT * pending_atr
                entry_time    = idx_arr[i]
                entry_idx     = i
                in_trade      = True
                pending_signal = 0
            elif bars_since >= RETEST_TIMEOUT_BARS:
                pending_signal = 0  # cancel
            continue

        # ── New signal ────────────────────────────────────────────────────────
        if sig != 0 and not in_trade:
            if ENTRY_MODE == 'close':
                entry_price   = c
                direction     = sig
                sl_dist       = compute_sl_distance(entry_price, direction,
                                                    atr_val, swl_arr[i], swh_arr[i])
                sl_distance   = sl_dist
                sl_price      = entry_price - direction * sl_dist
                tp_price      = entry_price + direction * RR_RATIO * sl_dist
                risk_amt      = capital * (RISK_PER_TRADE_PCT / 100)
                position_size = risk_amt / sl_dist
                trail_stop    = entry_price - direction * TRAILING_ATR_MULT * atr_val
                entry_time    = idx_arr[i]
                entry_idx     = i
                in_trade      = True

            elif ENTRY_MODE == 'retest':
                # Stage the pending entry
                pending_signal  = sig
                # Body range of the signal candle
                pending_body_hi = max(o, c)
                pending_body_lo = min(o, c)
                pending_sl      = atr_val * ATR_SL_MULTIPLIER
                pending_atr     = atr_val
                pending_bar     = i

    # Pad equity to full length
    while len(equity) < len(df):
        equity.append(capital)

    equity_series = pd.Series(equity[:len(df)], index=df.index)
    return trades, equity_series


print("🚀 Running backtest...")
trades, equity_curve = run_backtest(df_ltf)
print(f"✅ Backtest complete — {len(trades)} trades executed")

---
## Cell 8 — Performance Metrics

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PERFORMANCE ANALYTICS
# ══════════════════════════════════════════════════════════════════════════════

def compute_metrics(trades: list, equity: pd.Series, initial_capital: float) -> dict:
    if not trades:
        print("❌ No trades generated. Check parameters.")
        return {}

    df_t = pd.DataFrame(trades)
    pnl  = df_t['pnl_usd']

    wins  = df_t[df_t['result'] == 'WIN']
    loss  = df_t[df_t['result'] == 'LOSS']

    n_trades  = len(df_t)
    win_rate  = len(wins) / n_trades * 100
    avg_win   = wins['pnl_usd'].mean()  if len(wins)  > 0 else 0
    avg_loss  = loss['pnl_usd'].mean()  if len(loss)  > 0 else 0
    profit_factor = (wins['pnl_usd'].sum() / abs(loss['pnl_usd'].sum())
                     if abs(loss['pnl_usd'].sum()) > 0 else np.inf)

    total_return  = (equity.iloc[-1] / initial_capital - 1) * 100
    daily_returns = equity.pct_change().dropna()
    sharpe = (daily_returns.mean() / daily_returns.std() * np.sqrt(252 * 1440 /
              (len(equity) / len(df_t) if len(df_t) > 0 else 1))
              if daily_returns.std() > 0 else 0)

    # Max Drawdown
    roll_max  = equity.cummax()
    drawdown  = (equity - roll_max) / roll_max * 100
    max_dd    = drawdown.min()

    # Expectancy per trade
    expectancy = (win_rate/100 * avg_win) + ((1 - win_rate/100) * avg_loss)

    # Calmar
    calmar = total_return / abs(max_dd) if max_dd != 0 else np.inf

    # Average bars held
    avg_bars = df_t['bars_held'].mean()

    metrics = {
        'Total Trades'       : n_trades,
        'Win Rate (%)'       : round(win_rate, 2),
        'Profit Factor'      : round(profit_factor, 3),
        'Expectancy (USD)'   : round(expectancy, 2),
        'Total Return (%)'   : round(total_return, 2),
        'Max Drawdown (%)'   : round(max_dd, 2),
        'Sharpe Ratio'       : round(sharpe, 3),
        'Calmar Ratio'       : round(calmar, 3),
        'Avg Win (USD)'      : round(avg_win, 2),
        'Avg Loss (USD)'     : round(avg_loss, 2),
        'Avg W/L Ratio'      : round(abs(avg_win / avg_loss), 3) if avg_loss != 0 else np.inf,
        'Final Capital (USD)': round(equity.iloc[-1], 2),
        'Avg Bars Held'      : round(avg_bars, 1),
        'Long Trades'        : (df_t['direction'] == 'LONG').sum(),
        'Short Trades'       : (df_t['direction'] == 'SHORT').sum(),
        'TP Exits'           : (df_t['exit_reason'] == 'TP').sum(),
        'SL Exits'           : df_t['exit_reason'].str.contains('SL').sum(),
    }

    return metrics, df_t


result = compute_metrics(trades, equity_curve, INITIAL_CAPITAL)

if result:
    metrics, trades_df = result

    print("\n" + "═"*55)
    print("  📊  STRATEGY PERFORMANCE REPORT")
    print("═"*55)
    col_w = 28
    for k, v in metrics.items():
        bar = ""
        if k == 'Win Rate (%)':
            filled = int(v / 5)
            bar = " │ " + "█" * filled + "░" * (20 - filled) + f" {v}%"
        print(f"  {k:<{col_w}} {str(v):>10}{bar}")
    print("═"*55)

    if SHOW_TRADE_LOG and len(trades_df) > 0:
        print(f"\n📋 TRADE LOG (first {min(MAX_TRADES_IN_LOG, len(trades_df))} of {len(trades_df)} trades)")
        display_cols = ['entry_time','direction','entry_price','exit_price',
                        'sl_distance','pnl_usd','result','exit_reason','bars_held']
        print(trades_df[display_cols].head(MAX_TRADES_IN_LOG).to_string(index=False))

---
## Cell 9 — Visualizations

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  VISUALIZATIONS
# ══════════════════════════════════════════════════════════════════════════════

if not trades:
    print("No trades to visualize.")
else:
    fig = plt.figure(figsize=(20, 22))
    fig.patch.set_facecolor('#0d1117')
    gs  = fig.add_gridspec(4, 3, hspace=0.45, wspace=0.35)

    CYAN   = '#00d4ff'
    GREEN  = '#00ff88'
    RED    = '#ff4466'
    YELLOW = '#ffdd57'
    GRAY   = '#8b949e'
    BG     = '#0d1117'
    PANEL  = '#161b22'

    def style_ax(ax, title):
        ax.set_facecolor(PANEL)
        ax.tick_params(colors=GRAY, labelsize=8)
        ax.set_title(title, color=CYAN, fontsize=10, fontweight='bold', pad=8)
        for spine in ax.spines.values():
            spine.set_edgecolor('#30363d')
        ax.grid(True, color='#21262d', linewidth=0.5, alpha=0.7)

    # ── 1. Equity Curve (span top row) ────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    ec  = equity_curve.dropna()
    ax1.plot(ec.index, ec.values, color=CYAN, linewidth=1.2, zorder=3)
    ax1.fill_between(ec.index, INITIAL_CAPITAL, ec.values,
                     where=ec.values >= INITIAL_CAPITAL, alpha=0.15, color=GREEN)
    ax1.fill_between(ec.index, INITIAL_CAPITAL, ec.values,
                     where=ec.values < INITIAL_CAPITAL,  alpha=0.15, color=RED)
    ax1.axhline(INITIAL_CAPITAL, color=GRAY, linewidth=0.8, linestyle='--', alpha=0.6)

    # Mark trade entries on equity curve
    for t in trades:
        color = GREEN if t['result'] == 'WIN' else RED
        if t['entry_time'] in ec.index:
            ax1.axvline(t['entry_time'], color=color, alpha=0.15, linewidth=0.4)

    style_ax(ax1, f'Equity Curve — Initial: ${INITIAL_CAPITAL:,} | Final: ${equity_curve.iloc[-1]:,.0f}')
    ax1.set_ylabel('Capital (USD)', color=GRAY)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    # ── 2. Drawdown ────────────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, :])
    dd  = (equity_curve - equity_curve.cummax()) / equity_curve.cummax() * 100
    ax2.fill_between(dd.index, dd.values, 0, color=RED, alpha=0.5)
    ax2.plot(dd.index, dd.values, color=RED, linewidth=0.8)
    style_ax(ax2, f'Drawdown (%)  │  Max DD: {dd.min():.2f}%')
    ax2.set_ylabel('Drawdown %', color=GRAY)

    # ── 3. PnL Distribution ────────────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[2, 0])
    wins_pnl  = [t['pnl_usd'] for t in trades if t['result'] == 'WIN']
    loss_pnl  = [t['pnl_usd'] for t in trades if t['result'] == 'LOSS']
    if wins_pnl:
        ax3.hist(wins_pnl, bins=30, color=GREEN, alpha=0.7, label=f'Wins ({len(wins_pnl)})')
    if loss_pnl:
        ax3.hist(loss_pnl, bins=30, color=RED,   alpha=0.7, label=f'Losses ({len(loss_pnl)})')
    ax3.axvline(0, color=YELLOW, linewidth=1.2, linestyle='--')
    ax3.legend(fontsize=8, facecolor=PANEL, edgecolor='none', labelcolor=GRAY)
    style_ax(ax3, 'PnL Distribution (USD)')

    # ── 4. Win/Loss Pie ────────────────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[2, 1])
    n_w = len(wins_pnl); n_l = len(loss_pnl)
    ax4.pie([n_w, n_l],
            labels=[f'Win\n{n_w}', f'Loss\n{n_l}'],
            colors=[GREEN, RED], autopct='%1.1f%%',
            textprops={'color': GRAY, 'fontsize': 9},
            wedgeprops={'linewidth': 0})
    style_ax(ax4, 'Win / Loss Ratio')

    # ── 5. Cumulative PnL per trade ────────────────────────────────────────────
    ax5 = fig.add_subplot(gs[2, 2])
    cum_pnl = np.cumsum([t['pnl_usd'] for t in trades])
    colors  = [GREEN if p > 0 else RED for p in [t['pnl_usd'] for t in trades]]
    ax5.bar(range(len(trades)), [t['pnl_usd'] for t in trades], color=colors, alpha=0.8, width=0.7)
    ax5.plot(range(len(trades)), cum_pnl, color=CYAN, linewidth=1.5, label='Cumulative PnL')
    ax5.axhline(0, color=GRAY, linewidth=0.8, linestyle='--')
    ax5.legend(fontsize=8, facecolor=PANEL, edgecolor='none', labelcolor=GRAY)
    style_ax(ax5, 'Trade PnL + Cumulative')
    ax5.set_xlabel('Trade #', color=GRAY, fontsize=8)

    # ── 6. Bars Held Distribution ──────────────────────────────────────────────
    ax6 = fig.add_subplot(gs[3, 0])
    bars_held = [t['bars_held'] for t in trades]
    ax6.hist(bars_held, bins=min(40, len(set(bars_held))), color=CYAN, alpha=0.75, edgecolor='none')
    ax6.axvline(np.mean(bars_held), color=YELLOW, linewidth=1.5, linestyle='--',
                label=f'Mean = {np.mean(bars_held):.1f}')
    ax6.legend(fontsize=8, facecolor=PANEL, edgecolor='none', labelcolor=GRAY)
    style_ax(ax6, f'Bars Held Distribution ({LOWER_TF} bars)')

    # ── 7. Long vs Short breakdown ─────────────────────────────────────────────
    ax7 = fig.add_subplot(gs[3, 1])
    for direction, color in [('LONG', GREEN), ('SHORT', RED)]:
        subset = [t for t in trades if t['direction'] == direction]
        pnl_sub = [t['pnl_usd'] for t in subset]
        if pnl_sub:
            ax7.hist(pnl_sub, bins=20, color=color, alpha=0.65, label=f'{direction} ({len(subset)})')
    ax7.axvline(0, color=YELLOW, linewidth=1.2, linestyle='--')
    ax7.legend(fontsize=8, facecolor=PANEL, edgecolor='none', labelcolor=GRAY)
    style_ax(ax7, 'PnL by Direction')

    # ── 8. Monthly PnL ─────────────────────────────────────────────────────────
    ax8 = fig.add_subplot(gs[3, 2])
    trades_df_plot = pd.DataFrame(trades)
    trades_df_plot['entry_time'] = pd.to_datetime(trades_df_plot['entry_time'], utc=True)
    trades_df_plot['month'] = trades_df_plot['entry_time'].dt.to_period('M')
    monthly = trades_df_plot.groupby('month')['pnl_usd'].sum()
    bar_colors = [GREEN if v >= 0 else RED for v in monthly.values]
    ax8.bar(range(len(monthly)), monthly.values, color=bar_colors, alpha=0.85, width=0.7)
    ax8.set_xticks(range(len(monthly)))
    ax8.set_xticklabels([str(p) for p in monthly.index], rotation=45, fontsize=7, color=GRAY)
    ax8.axhline(0, color=GRAY, linewidth=0.8)
    style_ax(ax8, 'Monthly PnL (USD)')

    plt.suptitle(
        f'Wickless Candle Scalping Strategy  │  {LOWER_TF}/{HIGHER_TF}  │  {SYMBOL}  │  '
        f'Entry: {ENTRY_MODE}  │  RR: {RR_RATIO}:1  │  Disp: {USE_DISPLACEMENT}',
        color=CYAN, fontsize=13, fontweight='bold', y=0.995
    )

    plt.savefig('wickless_strategy_report.png', dpi=150, bbox_inches='tight',
                facecolor=BG, edgecolor='none')
    plt.show()
    print("\n📊 Chart saved to wickless_strategy_report.png")

---
## Cell 10 — Signal Quality Analysis

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  SIGNAL QUALITY ANALYSIS
#  Answers: do wickless candles have directional predictability?
#  Uses Spearman correlation of signal features vs forward returns
# ══════════════════════════════════════════════════════════════════════════════

print("📐 Signal Quality Analysis (Spearman ρ — features vs forward returns)\n")

horizons = [1, 3, 5, 10, 20]  # bars forward
feature_cols = ['b2r', 'bottom_wick', 'top_wick', 'atr']

results_rows = []
for h in horizons:
    fwd_ret = df_ltf['Close'].pct_change(h).shift(-h)
    for feat in feature_cols:
        if feat not in df_ltf.columns:
            continue
        valid = df_ltf[feat].notna() & fwd_ret.notna()
        rho, pval = spearmanr(df_ltf[feat][valid], fwd_ret[valid])
        results_rows.append({
            'Feature': feat,
            'Horizon (bars)': h,
            'Spearman ρ': round(rho, 4),
            'p-value': round(pval, 4),
            'Significant': '✅' if pval < 0.05 else '❌'
        })

spearman_df = pd.DataFrame(results_rows)
print(spearman_df.to_string(index=False))

# ── Wickless candle forward return statistics ─────────────────────────────────
print("\n" + "─"*55)
print("Forward Return Distribution — Wickless Candle Bars Only")
print("─"*55)

for direction_label, mask_col in [('BULLISH wickless', 'bull_wickless'),
                                   ('BEARISH wickless', 'bear_wickless')]:
    mask = df_ltf[mask_col]
    if USE_DISPLACEMENT:
        mask = mask & df_ltf['displaced']
    for h in [1, 5, 10]:
        fwd = df_ltf['Close'].pct_change(h).shift(-h)
        subset = fwd[mask].dropna() * 100
        if len(subset) < 10:
            continue
        t_stat, p = ttest_1samp(subset, 0)
        print(f"{direction_label:20s} | fwd={h:2d}bars | "
              f"n={len(subset):4d} | mean={subset.mean():+.4f}% | "
              f"t={t_stat:+.2f} | p={p:.4f} {'✅' if p < 0.05 else '❌'}")

---
## Cell 11 — Parameter Sensitivity Sweep

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PARAMETER SENSITIVITY SWEEP
#  Test different displacement thresholds and wick tolerances.
#  Quick grid search — does NOT change global parameters.
# ══════════════════════════════════════════════════════════════════════════════

print("🔬 Sensitivity Sweep — Displacement Threshold × Wick Tolerance\n")

disp_thresholds  = [0.60, 0.70, 0.75, 0.80, 0.85, 0.90]
wick_tolerances  = [0.0, 0.5, 1.0, 2.0, 5.0]          # USD

sweep_rows = []

for disp_t in disp_thresholds:
    for wick_t in wick_tolerances:
        # Recompute signals on a temporary df
        tmp = df_ltf.copy()
        bull_wl_t, bear_wl_t, _, _ = detect_wickless(tmp, wick_t)
        tmp['bull_wickless'] = bull_wl_t
        tmp['bear_wickless'] = bear_wl_t
        tmp['displaced']     = tmp['b2r'] >= disp_t
        tmp = generate_signals(tmp)

        t_trades, t_equity = run_backtest(tmp)
        if not t_trades:
            sweep_rows.append({
                'Disp Thresh': disp_t, 'Wick Tol': wick_t,
                'N Trades': 0, 'Win Rate': 0, 'PF': 0,
                'Total Ret %': 0, 'Max DD %': 0
            })
            continue

        t_df = pd.DataFrame(t_trades)
        wins_ = t_df[t_df['result'] == 'WIN']
        loss_ = t_df[t_df['result'] == 'LOSS']
        pf = (wins_['pnl_usd'].sum() / abs(loss_['pnl_usd'].sum())
              if len(loss_) > 0 and abs(loss_['pnl_usd'].sum()) > 0 else np.inf)
        wr = len(wins_) / len(t_df) * 100
        ret = (t_equity.iloc[-1] / INITIAL_CAPITAL - 1) * 100
        dd  = ((t_equity - t_equity.cummax()) / t_equity.cummax() * 100).min()

        sweep_rows.append({
            'Disp Thresh': disp_t, 'Wick Tol': wick_t,
            'N Trades': len(t_df),
            'Win Rate': round(wr, 1),
            'PF': round(pf, 2),
            'Total Ret %': round(ret, 2),
            'Max DD %': round(dd, 2)
        })

sweep_df = pd.DataFrame(sweep_rows)

# Highlight best rows by Profit Factor
print(sweep_df.sort_values('PF', ascending=False).to_string(index=False))

# ── Heatmap: Win Rate by Disp Threshold × Wick Tolerance ─────────────────────
pivot_wr = sweep_df.pivot(index='Disp Thresh', columns='Wick Tol', values='Win Rate')
pivot_pf = sweep_df.pivot(index='Disp Thresh', columns='Wick Tol', values='PF')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

for ax, pivot, title, fmt in [
    (axes[0], pivot_wr, 'Win Rate (%) Heatmap', '.1f'),
    (axes[1], pivot_pf, 'Profit Factor Heatmap', '.2f')
]:
    sns.heatmap(pivot, ax=ax, annot=True, fmt=fmt, cmap='RdYlGn',
                linewidths=0.5, linecolor='#21262d',
                annot_kws={'size': 9})
    ax.set_facecolor('#161b22')
    ax.set_title(title, color='#00d4ff', fontsize=11, fontweight='bold')
    ax.set_xlabel('Wick Tolerance (USD)', color='#8b949e')
    ax.set_ylabel('Displacement Threshold', color='#8b949e')
    ax.tick_params(colors='#8b949e')

plt.tight_layout()
plt.savefig('sensitivity_heatmap.png', dpi=130, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("\n📊 Sensitivity heatmap saved.")

---
## Cell 12 — Walk-Forward Validation

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  WALK-FORWARD VALIDATION
#  Splits data into N equal folds, runs backtest on each fold independently.
#  Checks for out-of-sample consistency vs full-period result.
# ══════════════════════════════════════════════════════════════════════════════

N_FOLDS = 4  # Adjust as needed

print(f"🔁 Walk-Forward Validation — {N_FOLDS} folds\n")

fold_size = len(df_ltf) // N_FOLDS
fold_results = []

for fold in range(N_FOLDS):
    start_i = fold * fold_size
    end_i   = (fold + 1) * fold_size if fold < N_FOLDS - 1 else len(df_ltf)
    fold_df = df_ltf.iloc[start_i:end_i].copy()

    f_trades, f_equity = run_backtest(fold_df)
    if not f_trades:
        fold_results.append({'Fold': fold+1, 'N Trades': 0,
                             'Win Rate': 0, 'PF': 0, 'Return %': 0})
        continue

    f_df   = pd.DataFrame(f_trades)
    f_wins = f_df[f_df['result'] == 'WIN']
    f_loss = f_df[f_df['result'] == 'LOSS']
    f_pf   = (f_wins['pnl_usd'].sum() / abs(f_loss['pnl_usd'].sum())
              if len(f_loss) > 0 and abs(f_loss['pnl_usd'].sum()) > 0 else np.inf)
    f_wr   = len(f_wins) / len(f_df) * 100
    f_ret  = (f_equity.iloc[-1] / INITIAL_CAPITAL - 1) * 100

    period_start = fold_df.index[0]
    period_end   = fold_df.index[-1]

    fold_results.append({
        'Fold'     : fold + 1,
        'Period'   : f"{str(period_start)[:10]} → {str(period_end)[:10]}",
        'N Trades' : len(f_df),
        'Win Rate' : round(f_wr, 1),
        'PF'       : round(f_pf, 2),
        'Return %' : round(f_ret, 2)
    })
    print(f"  Fold {fold+1} | {str(period_start)[:10]} → {str(period_end)[:10]} | "
          f"Trades: {len(f_df):3d} | WR: {f_wr:.1f}% | PF: {f_pf:.2f} | Ret: {f_ret:+.2f}%")

fold_df_display = pd.DataFrame(fold_results)
print("\n" + fold_df_display.to_string(index=False))

# Consistency check
if len(fold_results) > 1:
    wrs = [r['Win Rate'] for r in fold_results if r['N Trades'] > 0]
    pfs = [r['PF'] for r in fold_results if r['N Trades'] > 0 and r['PF'] != np.inf]
    print(f"\n📐 Win Rate  — mean: {np.mean(wrs):.1f}%  std: {np.std(wrs):.1f}%")
    print(f"📐 Prof Fac  — mean: {np.mean(pfs):.2f}   std: {np.std(pfs):.2f}")
    if np.std(wrs) < 10:
        print("✅ Win rate is CONSISTENT across folds (std < 10%)")
    else:
        print("⚠️  Win rate is INCONSISTENT across folds — possible overfitting or regime sensitivity")

---
## Cell 13 — Wickless Candle Visual Sample

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  VISUAL SAMPLE — Plot last N candles with wickless candles marked
# ══════════════════════════════════════════════════════════════════════════════

PLOT_LAST_N_BARS = 200   # Change to inspect more/less context

sample = df_ltf.tail(PLOT_LAST_N_BARS).copy()
sample = sample.reset_index()

fig, (ax_price, ax_b2r) = plt.subplots(2, 1, figsize=(20, 9),
                                         gridspec_kw={'height_ratios': [3, 1]},
                                         sharex=True)
fig.patch.set_facecolor('#0d1117')

for ax in [ax_price, ax_b2r]:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    ax.grid(True, color='#21262d', linewidth=0.5, alpha=0.7)
    for sp in ax.spines.values():
        sp.set_edgecolor('#30363d')

# Draw candles
bar_width = 0.6
for i, row in sample.iterrows():
    is_bull = row['Close'] >= row['Open']
    color   = '#00ff88' if is_bull else '#ff4466'
    alpha   = 1.0

    # Body
    body_bot = min(row['Open'], row['Close'])
    body_top = max(row['Open'], row['Close'])
    ax_price.add_patch(mpatches.FancyBboxPatch(
        (i - bar_width/2, body_bot), bar_width, body_top - body_bot,
        boxstyle="square,pad=0", linewidth=0, facecolor=color, alpha=alpha, zorder=2
    ))
    # Wicks
    ax_price.plot([i, i], [row['Low'],  body_bot], color=color, linewidth=0.8, zorder=1)
    ax_price.plot([i, i], [body_top, row['High']], color=color, linewidth=0.8, zorder=1)

    # Mark wickless candles
    if row['bull_wickless'] and (not USE_DISPLACEMENT or row['displaced']):
        ax_price.scatter(i, row['Low'] - row['atr'] * 0.3, marker='^',
                         color='#00d4ff', s=60, zorder=5)
    if row['bear_wickless'] and (not USE_DISPLACEMENT or row['displaced']):
        ax_price.scatter(i, row['High'] + row['atr'] * 0.3, marker='v',
                         color='#ffdd57', s=60, zorder=5)

# Supertrend line (if available in sample)
# (HTF supertrend forwarded, approximate visualization)

# EMA line
ema_vals = sample['Close'].ewm(span=EMA_PERIOD, adjust=False).mean()
ax_price.plot(range(len(sample)), ema_vals.values, color='#ffdd57',
              linewidth=1.2, alpha=0.8, label=f'EMA({EMA_PERIOD})')

ax_price.legend(fontsize=9, facecolor='#161b22', edgecolor='none', labelcolor='#8b949e')
ax_price.set_title(
    f'Wickless Candle Detection — Last {PLOT_LAST_N_BARS} bars ({LOWER_TF})\n'
    f'🔵 = Bullish wickless (no bottom wick)  |  🟡 = Bearish wickless (no top wick)',
    color='#00d4ff', fontsize=10, fontweight='bold'
)
ax_price.set_ylabel('Price (USD)', color='#8b949e')

# B2R panel
b2r_vals = sample['b2r'].values
bar_colors_b2r = ['#00ff88' if sample['Close'].iloc[i] >= sample['Open'].iloc[i]
                  else '#ff4466' for i in range(len(sample))]
ax_b2r.bar(range(len(sample)), b2r_vals, color=bar_colors_b2r, alpha=0.7, width=0.7)
ax_b2r.axhline(DISPLACEMENT_THRESHOLD, color='#ffdd57', linewidth=1.2,
                linestyle='--', label=f'Threshold={DISPLACEMENT_THRESHOLD}')
ax_b2r.legend(fontsize=8, facecolor='#161b22', edgecolor='none', labelcolor='#8b949e')
ax_b2r.set_ylabel('B2R', color='#8b949e')
ax_b2r.set_title('Body-to-Range Ratio (Displacement)', color='#00d4ff', fontsize=9)
ax_b2r.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('wickless_candle_sample.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("📊 Sample chart saved.")

---
## Cell 14 — Summary & Next Steps

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  FINAL SUMMARY PRINT
# ══════════════════════════════════════════════════════════════════════════════

print("═"*60)
print("  🏁  RUN SUMMARY")
print("═"*60)
print(f"  Symbol         : {SYMBOL}")
print(f"  Period         : {df_ltf.index[0]} → {df_ltf.index[-1]}")
print(f"  Timeframes     : Entry={LOWER_TF}  |  Trend={HIGHER_TF}")
print(f"  Entry mode     : {ENTRY_MODE}")
print(f"  Wick tolerance : {WICK_TOLERANCE} USD")
print(f"  Displacement   : {'ON @ ' + str(DISPLACEMENT_THRESHOLD) if USE_DISPLACEMENT else 'OFF'}")
print(f"  Trend filter   : Supertrend({SUPERTREND_PERIOD},{SUPERTREND_MULT}) + "
      f"EMA({EMA_PERIOD}) slope = {'ON' if REQUIRE_EMA_CONFIRM else 'Supertrend only'}")
print(f"  SL method      : min(ATR×{ATR_SL_MULTIPLIER}, Swing lookback={SWING_LOOKBACK})")
print(f"  TP method      : {'Trailing ATR×' + str(TRAILING_ATR_MULT) if USE_TRAILING_STOP else 'Fixed RR ' + str(RR_RATIO) + ':1'}")
print("─"*60)
if trades:
    print(f"  Total trades   : {metrics['Total Trades']}")
    print(f"  Win rate       : {metrics['Win Rate (%)']}%")
    print(f"  Profit factor  : {metrics['Profit Factor']}")
    print(f"  Total return   : {metrics['Total Return (%)']}%")
    print(f"  Max drawdown   : {metrics['Max Drawdown (%)']}%")
    print(f"  Sharpe ratio   : {metrics['Sharpe Ratio']}")
    print(f"  Final capital  : ${metrics['Final Capital (USD)']:,}")
else:
    print("  ⚠️  No trades generated. Try relaxing WICK_TOLERANCE,")
    print("      lowering DISPLACEMENT_THRESHOLD, or checking date range.")
print("═"*60)

print("""
💡 SUGGESTED NEXT EXPERIMENTS:

  1. Test ENTRY_MODE = 'retest' — typically improves RR but reduces trade count
  2. Sweep WICK_TOLERANCE from 0 → 5 USD (see Cell 11 heatmap)
  3. Try USE_TRAILING_STOP = True for trending regimes
  4. Change HIGHER_TF to '1h' or '4h' for fewer but higher-quality signals
  5. Set REQUIRE_EMA_CONFIRM = False to see raw Supertrend-only effect
  6. Load 5-year local CSV and run Cell 12 walk-forward on full history
  7. Add session filter (e.g. UTC 13:00–21:00 for NY session only)
""")